# `collections` => deque, namedtuple, OrderedDict & ChainMap

| Tool | Purpose | Key members |
|---|---|---|
| `deque` | Fast appends and pops at **both ends** | `append`, `appendleft`, `pop`, `popleft`, `rotate`, `extend`, `extendleft`, `maxlen` |
| `namedtuple` | A tuple whose items have names | `_fields`, `_asdict()`, `_replace()`, `_make()` |
| `OrderedDict` | A dict with order-aware extras | `move_to_end()`, `popitem(last=False)` |
| `ChainMap` | Several dicts searched as one | `maps`, `new_child()`, `parents` |

```python
from collections import deque, namedtuple, OrderedDict, ChainMap
```

---

## `deque` (Double-Ended Queue)

A `deque` is optimized for adding and removing items at **both ends**.

| Method | Purpose |
|---|---|
| `append(x)` | Add to the right |
| `appendleft(x)` | Add to the left |
| `pop()` | Remove from the right |
| `popleft()` | Remove from the left |
| `extend(it)` | Add many to the right |
| `extendleft(it)` | Add many to the left (order is reversed) |
| `rotate(n)` | Rotate right by `n` (left if negative) |
| `clear()` | Remove everything |

### `maxlen`

`deque(maxlen=3)` keeps only the **last 3** items. Older items are discarded automatically.

### Queue and Stack

| Structure | Rule | Operations |
|---|---|---|
| **Queue** | FIFO: first in, first out | `append` + `popleft` |
| **Stack** | LIFO: last in, first out | `append` + `pop` |

### Important

* `list.pop(0)` is slow for big lists because every item shifts. `deque.popleft()` is fast.
* Indexing in the **middle** of a deque is slow. Use a list when you need random access.

---

## `namedtuple`

Creates a tuple subclass with **named fields**.

### Syntax

`Point = namedtuple("Point", ["x", "y"])`

| Feature | Example |
|---|---|
| Create | `p = Point(1, 2)` or `Point(x=1, y=2)` |
| Access by name | `p.x` |
| Access by index | `p[0]` |
| Unpack | `x, y = p` |
| To dict | `p._asdict()` |
| Copy with changes | `p._replace(x=10)` |
| Field names | `Point._fields` |
| Defaults | `namedtuple("P", ["x", "y"], defaults=[0])` |

### Important

* A namedtuple is **immutable**, like a tuple.
* Need types or methods? Use `typing.NamedTuple` (Part 10) or a dataclass (Part 9).

---

## `OrderedDict`

Since Python 3.7 a normal `dict` keeps insertion order. `OrderedDict` still adds:

* `move_to_end(key, last=True)` to reorder items.
* `popitem(last=False)` to remove the **oldest** item.
* Equality between two `OrderedDict` objects **checks the order**.

Typical use: a small cache that evicts the oldest entry.

---

## `ChainMap`

Groups several dictionaries into one **view**. A lookup checks each dictionary in order.

| Feature | Behavior |
|---|---|
| Lookup | First dictionary that has the key wins |
| Write, update, delete | Affects only the **first** dictionary |
| `maps` | The list of underlying dictionaries |
| `new_child()` | A new ChainMap with an empty first dictionary |
| `parents` | All dictionaries except the first |

Typical use: command-line options over environment settings over defaults.

## Source

https://docs.python.org/3/library/collections.html#collections.deque

https://docs.python.org/3/library/collections.html#collections.namedtuple

https://docs.python.org/3/library/collections.html#collections.OrderedDict

https://docs.python.org/3/library/collections.html#collections.ChainMap

In [ ]:
from collections import deque, namedtuple, OrderedDict, ChainMap

# deque: both ends
d = deque([1, 2, 3])
d.append(4)
d.appendleft(0)
print(d)                                   # deque([0, 1, 2, 3, 4])
print(d.pop(), d.popleft(), d)             # 4 0 deque([1, 2, 3])
d.extend([4, 5])
d.extendleft([-1, -2])                     # each item goes to the left, one by one
print(d)                                   # deque([-2, -1, 1, 2, 3, 4, 5])
d.rotate(2)
print(d)                                   # last two items move to the front
d.rotate(-2)
print(d)

# maxlen: keep only the last N items
last3 = deque(maxlen=3)
for n in range(6):
    last3.append(n)
print(last3)                               # deque([3, 4, 5], maxlen=3)

# Queue (FIFO) and stack (LIFO)
queue = deque()
queue.append("a"); queue.append("b"); queue.append("c")
print(queue.popleft(), queue.popleft())    # a b   (first in, first out)

stack = deque()
stack.append("a"); stack.append("b"); stack.append("c")
print(stack.pop(), stack.pop())            # c b   (last in, first out)

# namedtuple
Point = namedtuple("Point", ["x", "y"])
p = Point(1, 2)
print(p, p.x, p[1])
x, y = p
print(x, y, Point._fields)
print(p._asdict())
print(p._replace(x=10))
print(Point._make([5, 6]))
Point3 = namedtuple("Point3", ["x", "y", "z"], defaults=[0])
print(Point3(1, 2))                        # z defaults to 0
try:
    p.x = 5
except AttributeError as error:
    print("immutable:", type(error).__name__)

# OrderedDict
od = OrderedDict(a=1, b=2, c=3)
od.move_to_end("a")
print(list(od))                            # ['b', 'c', 'a']
print(od.popitem(last=False))              # ('b', 2): removes the oldest
print(OrderedDict(a=1, b=2) == OrderedDict(b=2, a=1))     # False: order matters
print(dict(a=1, b=2) == dict(b=2, a=1))                   # True

# ChainMap: command line > environment > defaults
defaults = {"color": "red", "user": "guest"}
environment = {"user": "ann"}
command_line = {"color": "blue"}
settings = ChainMap(command_line, environment, defaults)
print(settings["color"], settings["user"])          # blue ann
settings["debug"] = True                            # writes go to the first dict only
print(command_line)
print(len(settings.maps), list(settings.parents.maps[0]))